In [1]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 103.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers-5.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.5.5 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.9.0 which is incompatible.
unsloth-zoo 2026.5.3 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.9.0 which is incompatible.


In [2]:
!pip install unsloth trl

  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.9.0
    Uninstalling transformers-5.9.0:
      Successfully uninstalled transformers-5.9.0


In [3]:
from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig
import pandas as pd
import json
import torch
import re
import ast
import torch
from tqdm import tqdm
from transformers import AutoTokenizer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
import warnings
warnings.filterwarnings('ignore')

In [5]:
# ==========================================================
# Qwen2.5-7B-Instruct QLoRA fine-tuning for JSON macro extraction
# GPU: A100 80GB
# Library: unsloth
# ==========================================================
## СЛАВА КОЛАБУ ПРО

In [6]:
## ЗАДАЕМ ПАРАМЕТРЫ
LORA_DROPOUT = 0 ## СТАНДАРТ ДЛЯ UNSLOTH
WEIGHT_DECAY = 0.1 ## тестим 0.01, 0.05, 0.1

In [7]:
max_seq_length = 4096
load_in_4bit = False  ## У НАС МНОГО ПАМЯТИ
dtype = None # UNSLOTH САМ ПОЙМЕТ

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.5.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [8]:
## LORA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 32, ## 2r
    lora_dropout = LORA_DROPOUT, ## делаем побольше для того, чтобы модель не переобучалась -- подумать
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)


Unsloth 2026.5.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [9]:
## НАСТРАИВАЕМ PAD ТОКЕН
if tokenizer.pad_token is None:
    if tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        tokenizer.add_special_tokens({'pad_token': '<|endoftext|>'})

## СИНХРОНИРУЕМ CONFIG МОДЕЛИ С ТОКЕНАЙЗЕРОМ
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
if hasattr(tokenizer, "bos_token_id"):
    model.config.bos_token_id = tokenizer.bos_token_id

## СИНХРОНИРУЕМ generation_config
if hasattr(model, "generation_config") and model.generation_config is not None:
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    model.generation_config.bos_token_id = model.config.bos_token_id

In [10]:
## БЕРЕМ ВЕСЬ ДАТАСЕТ
data_2000 = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vRmFclgtiHBuF-L7VetTy4NtyFzozR_5hXAh5XjkaiRGaqnnRaGu6_AvjsN1ZNzk14VX88_BSBDEK0r/pub?output=csv")
## df_2000_all_new.csv
data_2000

,процентная ставка,ВВП,инфляция,безработица,капитал,инвестиции,производство,потребление,численность рабочей силы,сбережения,...,доходы населения,валютный курс,импорт,экспорт,государственные расходы,государственный долг,дефицит бюджета,doc_id,chunk_id,original_text
0,not stated,not stated,not stated,not stated,down,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,1,0,"""Индекс Мосбиржи -3%. Это самое большое дневно..."
1,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,2,0,"""​​Циан взлетел на апелляции Крупнейший в Росс..."
2,not stated,down,up,not stated,not stated,not stated,down,not stated,not stated,not stated,...,down,not stated,not stated,down,not stated,not stated,up,2,1,"""С 5 февраля ЕС вводит эмбарго на российские п..."
3,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,up,3,0,Госбюджет РФ рискует недополучить еще больше н...
4,not stated,not stated,not stated,not stated,up,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,4,0,"""Новый отчёт Сбера (SBER): чистая прибыль за 1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1801,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,1488,0,• Объем торгов металлами в феврале составил 61...
1802,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,1489,0,"""​​Страхованию инвестиций на ИИС — быть В дека..."
1803,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,up,not stated,not stated,not stated,not stated,not stated,1489,1,"""​​Что означает 100 рублей за доллар для префо..."
1804,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,up,not stated,not stated,not stated,not stated,not stated,1491,0,В начале октября валютный курс тестирует сопро...


In [11]:
## КОСЯКА ЗДЕСЬ ТОЧНО НЕ БЫЛО)
data_2000 = data_2000[data_2000['doc_id'] != 'up']
data_2000['doc_id'] = data_2000['doc_id'].astype('int')

In [12]:
## ИДЕАЛЬНОЕ РАЗБИЕНИЕ
test = data_2000[(data_2000['doc_id'] > 1234)]
val = data_2000[(data_2000['doc_id'] > 1000) & (data_2000['doc_id'] <= 1234)]
train = data_2000[data_2000['doc_id'] <= 1000]

In [13]:
## убираем данные где везде not stated
train = train[((train == 'not stated').sum(axis=1) != 18)]
val = val[((val == 'not stated').sum(axis=1) != 18)]

In [14]:
train.shape, val.shape, test.shape

((948, 21), (235, 21), (268, 21))

In [15]:
## ДЕЛАЕМ ДАТЕСЕТЫ ДЛЯ УДОБСТВА И ПЕРЕКЛАДЫВАЕМ В JSON
train_new = [{"json_target": str(dict(train.iloc[i, :18])), "text": train.iloc[i, -1]} for i in range(len(train))]
train_new = pd.DataFrame(train_new)

val_new = [{"json_target": str(dict(val.iloc[i, :18])), "text": val.iloc[i, -1]} for i in range(len(val))]
val_new = pd.DataFrame(val_new)

test_new = [{"json_target": str(dict(test.iloc[i, :18])), "text": test.iloc[i, -1]} for i in range(len(test))]
test_new = pd.DataFrame(test_new)

In [16]:
## ЕЩЕ ТРИ КРУТЫХ ДАТАСЕТА
train_df = train_new
val_df = val_new
test_df = test_new

In [23]:
## ДЕЛАЕМ TRAIN И VAL ДЛЯ ОБУЧЕНИЯ
def format_dataset_for_completion_loss(row):
    system_prompt = "Ты — экспертный макроэкономический аналитик."
    user_prompt = ( "Твоя задача — анализировать новости "
        "и определять динамику 18 показателей. Оценивай прямые упоминания и логические следствия. "
        "Допустимые значения изменений ТОЛЬКО ТАКИЕ: 'up', 'down', 'not stated'. "
        "Ответь строго в формате JSON без вступлений и пояснений: "
        "{"
        "\"процентная ставка\": \"тип изменения\","
        "\"ВВП\": \"тип изменения\","
        "\"инфляция\": \"тип изменения\","
        "\"безработица\": \"тип изменения\","
        "\"капитал\": \"тип изменения\","
        "\"инвестиции\": \"тип изменения\","
        "\"производство\": \"тип изменения\","
        "\"потребление\": \"тип изменения\","
        "\"численность рабочей силы\": \"тип изменения\","
        "\"сбережения\": \"тип изменения\","
        "\"заработные платы\": \"тип изменения\","
        "\"доходы населения\": \"тип изменения\","
        "\"валютный курс\": \"тип изменения\","
        "\"импорт\": \"тип изменения\","
        "\"экспорт\": \"тип изменения\","
        "\"государственные расходы\": \"тип изменения\","
        "\"государственный долг\": \"тип изменения\","
        "\"дефицит бюджета\": \"тип изменения\""
        "} "
        f"Текст новости: {row['text']}"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    completion_text = f"{row['json_target']}<|im_end|>\n"

    return {
        "prompt": prompt_text,
        "completion": completion_text
    }
train_dataset = Dataset.from_pandas(train_df).map(format_dataset_for_completion_loss)
val_dataset = Dataset.from_pandas(val_df).map(format_dataset_for_completion_loss)

train_dataset = train_dataset.remove_columns([c for c in train_dataset.column_names if c not in ['prompt', 'completion']])
val_dataset = val_dataset.remove_columns([c for c in val_dataset.column_names if c not in ['prompt', 'completion']])

Map:   0%|          | 0/948 [00:00<?, ? examples/s]

Map:   0%|          | 0/235 [00:00<?, ? examples/s]

In [34]:
## ДЕЛАЕМ TEST ДЛЯ ПРЕДСКАЗАНИЯ
def format_test_row(row):
    system_prompt = "Ты — экспертный макроэкономический аналитик."
    user_prompt =( "Твоя задача — анализировать новости "
        "и определять динамику 18 показателей. Оценивай прямые упоминания и логические следствия. "
        "Допустимые значения изменений ТОЛЬКО ТАКИЕ: 'up', 'down', 'not stated'. "
        "Ответь строго в формате JSON без вступлений и пояснений: "
        "{"
        "\"процентная ставка\": \"тип изменения\","
        "\"ВВП\": \"тип изменения\","
        "\"инфляция\": \"тип изменения\","
        "\"безработица\": \"тип изменения\","
        "\"капитал\": \"тип изменения\","
        "\"инвестиции\": \"тип изменения\","
        "\"производство\": \"тип изменения\","
        "\"потребление\": \"тип изменения\","
        "\"численность рабочей силы\": \"тип изменения\","
        "\"сбережения\": \"тип изменения\","
        "\"заработные платы\": \"тип изменения\","
        "\"доходы населения\": \"тип изменения\","
        "\"валютный курс\": \"тип изменения\","
        "\"импорт\": \"тип изменения\","
        "\"экспорт\": \"тип изменения\","
        "\"государственные расходы\": \"тип изменения\","
        "\"государственный долг\": \"тип изменения\","
        "\"дефицит бюджета\": \"тип изменения\""
        "} "
        f"Текст новости: {row['text']}"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    prompt_for_model = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    return {
        "prompt": prompt_for_model,
        "true_target": row['json_target']
    }
hf_test_dataset = Dataset.from_pandas(test_df)
test_dataset = hf_test_dataset.map(format_test_row)

Map:   0%|          | 0/268 [00:00<?, ? examples/s]

In [24]:
# Настраиваем трейнер
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    max_seq_length = max_seq_length,
    packing = False,  # ДЛЯ РАБОТЫ С JSON-ОМ НЕ НАДО
    args = SFTConfig(
        completion_only_loss = True,
        per_device_train_batch_size = 24, ## было 48, сделал меньше чтобы моделька медленнее сходилась
        gradient_accumulation_steps = 1,
        num_train_epochs = 2, ## БЕРЕМ МЕНЬШЕ ЭПОХ, Т.К ДАЛЬШЕ ПЕРЕОБУЧАЕТСЯ
        learning_rate = 1e-5, ## берем меньше, иначе моделька выдает только N/S
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = WEIGHT_DECAY,
        output_dir = "outputs",
        save_strategy = "epoch",
        eval_strategy = "epoch",
        metric_for_best_model = "eval_loss",
        load_best_model_at_end = True,
        save_total_limit = 3,
        report_to = "none",
        max_grad_norm=1,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=16):   0%|          | 0/948 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=16):   0%|          | 0/235 [00:00<?, ? examples/s]

In [25]:
# СТАРТУЕМ
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 948 | Num Epochs = 2 | Total steps = 80
O^O/ \_/ \    Batch size per device = 24 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (24 x 1 x 1) = 24
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.024794,0.021584
2,0.016478,0.016257


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-40/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-80/tokenizer_config.json.


TrainOutput(global_step=80, training_loss=0.048678172007203105, metrics={'train_runtime': 399.8263, 'train_samples_per_second': 4.742, 'train_steps_per_second': 0.2, 'total_flos': 6.956066454453043e+16, 'train_loss': 0.048678172007203105, 'epoch': 2.0})

In [44]:
### СОХРАНИМ
model.save_pretrained("qwen_macro_lora")
tokenizer.save_pretrained("qwen_macro_lora")

Unsloth: Restored added_tokens_decoder metadata in qwen_macro_lora/tokenizer_config.json.


('qwen_macro_lora/tokenizer_config.json',
 'qwen_macro_lora/chat_template.jinja',
 'qwen_macro_lora/tokenizer.json')

In [45]:
## сначала в zip
!zip -r qwen_macro_lora.zip ./qwen_macro_lora

  adding: qwen_macro_lora/ (stored 0%)
  adding: qwen_macro_lora/adapter_config.json (deflated 59%)
  adding: qwen_macro_lora/chat_template.jinja (deflated 71%)
  adding: qwen_macro_lora/README.md (deflated 65%)
  adding: qwen_macro_lora/tokenizer_config.json (deflated 89%)
  adding: qwen_macro_lora/adapter_model.safetensors (deflated 8%)
  adding: qwen_macro_lora/tokenizer.json (deflated 81%)


In [46]:
## сохраним в облако
!curl -F "file=@./qwen_macro_lora.zip" https://store1.gofile.io/uploadFile

{"data":{"createTime":1779294503,"downloadPage":"https://gofile.io/d/AnZ6dC","guestToken":"BCKKijLRj9xTC96dgrjnsnxRCOscDZyv","id":"f28ddaa9-6856-447f-8edc-26dc7246ed84","md5":"37221655e5d20884187492230eb73bd7","mimetype":"application/zip","modTime":1779294503,"name":"qwen_macro_lora.zip","parentFolder":"9f75bdd6-5530-40c2-85bc-3d1c4ee96b70","parentFolderCode":"AnZ6dC","servers":["store1"],"size":150518756,"type":"file"},"status":"ok"}


## ТЕСТ НОВОЙ МОДЕЛИ

In [27]:
## ЕЩЕ РАЗ ДЕЛАЕМ ТЕСТОВЫЙ ДАТАСЕТ
test_data = list(pd.DataFrame(test_dataset)['text'])

### ЧУТЬ ПО-ДРУГОМУ БУДЕМ ТЕСТИТЬ МОДЕЛЬКУ, Т.К В TRAIN И VAL ЕСТЬ ASSISTANT

In [26]:
def find_metrics(responces, df_to_check):
    expected_keys = [
        "процентная ставка", "ввп", "инфляция", "безработица", "капитал",
        "инвестиции", "производство", "потребление", "численность рабочей силы",
        "сбережения", "заработные платы", "доходы населения", "валютный курс",
        "импорт", "экспорт", "государственные расходы", "государственный долг",
        "дефицит бюджета"
    ]

    total_accuracy = 0
    valid_json_count = 0
    up_down_accuracy = 0
    not_up_down_total = 0

    for i, pred_raw in enumerate(responces):
        try:
            clean_pred = pred_raw.strip()

            json_match = re.search(r'\{.*\}', clean_pred, re.DOTALL)
            if not json_match:
                print(f"Документ {i+1}: JSON не найден")
                continue

            json_str = json_match.group()

            try:
                raw_pred_json = json.loads(json_str)
            except json.JSONDecodeError:
                raw_pred_json = ast.literal_eval(json_str)

            pred_json = {
                str(k).lower().replace('капital', 'капитал').replace('kapital', 'капитал').strip(): v
                for k, v in raw_pred_json.items()
            }

            row = df_to_check.iloc[i]
            target_raw = row['json_target']

            if isinstance(target_raw, str):
                try:
                    target_dict = json.loads(target_raw)
                except:
                    target_dict = ast.literal_eval(target_raw)
            else:
                target_dict = target_raw

            target_dict = {str(k).lower().strip(): v for k, v in target_dict.items()}

            correct_in_doc = 0
            correct_up_down = 0
            count_up_down = 0

            for entity in expected_keys:
                model_val = str(pred_json.get(entity, "not stated")).lower().strip()
                true_val = str(target_dict.get(entity, "not stated")).lower().strip()

                if model_val == true_val:
                    correct_in_doc += 1

                if true_val != 'not stated':
                    count_up_down += 1
                    if model_val == true_val:
                        correct_up_down += 1

            if count_up_down > 0:
                up_down_accuracy += (correct_up_down / count_up_down)
            else:
                not_up_down_total += 1

            total_accuracy += (correct_in_doc / len(expected_keys))
            valid_json_count += 1

        except Exception as e:
            print(f"Ошибка в документе {i+1}: {type(e).__name__} - {e}")

    ## ИТОГОВЫЕ ЦИФРЫ
    k = len(responces)
    print("\n" + "="*30)
    print(f"ОБРАБОТАНО УСПЕШНО: {valid_json_count} из {k}")
    if k > 0:
        print(f"ОБЩАЯ ACCURACY (по всем ключам): {total_accuracy/k:.4f}")
        divisor = (k - not_up_down_total)
        if divisor > 0:
            print(f"ACCURACY (только UP/DOWN): {up_down_accuracy/divisor:.4f}")

In [37]:
test_dataset = pd.DataFrame(test_dataset)

## СЧИТАЕМ МЕТРИКИ ДЛЯ ДООБУЧЕННОЙ МОДЕЛИ НА ТЕСТЕ

In [43]:
## ИНФЕРЕНС
FastLanguageModel.for_inference(model)
model.eval()

responses = []
true_targets = []

print("\n--- Запуск инференса на тестовой выборке ---")

for i in tqdm(range(len(test_dataset))):
    prompt = test_dataset.iloc[i]['prompt']
    target = test_dataset.iloc[i]['true_target']

    # ТОКЕНИЗИРУЕМ ПРОМПТ
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            max_length=None,
            do_sample=False,
            temperature=None,
            top_p=None,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    # ЧИСТЫЙ ОТВЕТ
    prompt_len = inputs.input_ids.shape[1]
    completion_ids = generated_ids[0][prompt_len:]

    # ДЕКОДИРУЕМ
    response_text = tokenizer.decode(completion_ids, skip_special_tokens=True).strip()

    responses.append(response_text)
    true_targets.append(target)

    if i % 10 == 9:
        print(f"\n📊 --- МЕТРИКИ ДЛЯ i = {i + 1} ---")

        current_df_slice = test_dataset.iloc[:i+1]
        find_metrics(responses, df_to_check=current_df_slice)

        # СЧИТАЕМ ГДЕ ВСЕ ЗНАЧЕНИЯ 'not stated'
        only_resp_sum = 0
        for j in range(len(responses)):
            try:
                parsed_json = json.loads(responses[j])

                ## ВСЕ ЗНАЧЕНИЯ - 'not stated'
                if all(str(val).lower().strip() == 'not stated' for val in parsed_json.values()):
                    only_resp_sum += 1

            except (json.JSONDecodeError, TypeError, AttributeError):
                pass

        share_empty = only_resp_sum / len(responses)
        print(f'Доля ответов только с "not stated": {share_empty:.4f}')
        print("--------------------------------------\n")


--- Запуск инференса на тестовой выборке ---


  4%|▎         | 10/268 [01:32<39:44,  9.24s/it]


📊 --- МЕТРИКИ ДЛЯ i = 10 ---

ОБРАБОТАНО УСПЕШНО: 10 из 10
ОБЩАЯ ACCURACY (по всем ключам): 0.9667
ACCURACY (только UP/DOWN): 0.7292
Доля ответов только с "not stated": 0.0000
--------------------------------------



  7%|▋         | 20/268 [03:04<37:53,  9.17s/it]


📊 --- МЕТРИКИ ДЛЯ i = 20 ---

ОБРАБОТАНО УСПЕШНО: 20 из 20
ОБЩАЯ ACCURACY (по всем ключам): 0.9694
ACCURACY (только UP/DOWN): 0.7444
Доля ответов только с "not stated": 0.0000
--------------------------------------



...


📊 --- МЕТРИКИ ДЛЯ i = 160 ---

ОБРАБОТАНО УСПЕШНО: 160 из 160
ОБЩАЯ ACCURACY (по всем ключам): 0.9372
ACCURACY (только UP/DOWN): 0.7472
Доля ответов только с "not stated": 0.0000
--------------------------------------



 63%|██████▎   | 170/268 [25:59<14:51,  9.09s/it]


📊 --- МЕТРИКИ ДЛЯ i = 170 ---

ОБРАБОТАНО УСПЕШНО: 170 из 170
ОБЩАЯ ACCURACY (по всем ключам): 0.9333
ACCURACY (только UP/DOWN): 0.7456
Доля ответов только с "not stated": 0.0000
--------------------------------------



 64%|██████▍   | 172/268 [26:19<14:41,  9.18s/it]


KeyboardInterrupt: 

## ИСПОЛЬЗОВАНИЕ РАСПАКОВАННОЙ МОДЕЛИ

In [ ]:
# Распаковываем
!unzip qwen_macro_lora.zip

In [ ]:
from peft import PeftModel

model_id ="Qwen/Qwen2.5-7B-Instruct" # Исходная модель
adapter_path = "qwen_macro_lora" # Ваша папка

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, adapter_path)

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

In [ ]:
### ДЕЛАЕМ ДАТАСЕТ ДЛЯ ОБУЧЕНИЯ
def prepare_train_dataset(df, tokenizer):
    """Функция для полной подготовки датасета к обучению"""

    def formatting_prompts_func(examples):
        instructions = []
        for text, target in zip(examples["text"], examples["json_target"]):

            user_prompt = (
                "Твоя задача — анализировать новости "
                "и определять динамику 18 показателей. Оценивай прямые упоминания и логические следствия. "
                "Допустимые значения изменений ТОЛЬКО ТАКИЕ: 'up', 'down', 'not stated'. "
                "Ответь строго в формате JSON без вступлений и пояснений: "
                "{"
                "\"процентная ставка\": \"тип изменения\","
                "\"ВВП\": \"тип изменения\","
                "\"инфляция\": \"тип изменения\","
                "\"безработица\": \"тип изменения\","
                "\"капитал\": \"тип изменения\","
                "\"инвестиции\": \"тип изменения\","
                "\"производство\": \"тип изменения\","
                "\"потребление\": \"тип изменения\","
                "\"численность рабочей силы\": \"тип изменения\","
                "\"сбережения\": \"тип изменения\","
                "\"заработные платы\": \"тип изменения\","
                "\"доходы населения\": \"тип изменения\","
                "\"валютный курс\": \"тип изменения\","
                "\"импорт\": \"тип изменения\","
                "\"экспорт\": \"тип изменения\","
                "\"государственные расходы\": \"тип изменения\","
                "\"государственный долг\": \"тип изменения\","
                "\"дефицит бюджета\": \"тип изменения\""
                "} "
                f"Текст новости: {text}"
            )

            messages = [
                {"role": "system", "content": "Ты — экспертный макроэкономический аналитик."},
                {"role": "user", "content": user_prompt}
            ]

            # Применяем CHAT TEMPLATE
            formatted_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            if isinstance(target, dict):
                target_str = json.dumps(target, ensure_ascii=False)
            else:
                target_str = str(target).strip()

            # СКЛЕИВАЕМ Промпт + Ответ + EOS токен
            full_text = f"{formatted_text}{target_str}{tokenizer.eos_token}"
            instructions.append(full_text)

        return {"text": instructions}

    # HuggingFace Dataset
    dataset = Dataset.from_pandas(df[['text', 'json_target']])

    dataset = dataset.map(formatting_prompts_func, batched=True)

    return dataset

In [ ]:
def prepare_prompt_completion(example):
    # Разделяем текст по тегу начала ответа ассистента
    parts = example['text'].split("<|im_start|>assistant\n")
    prompt = parts[0] + "<|im_start|>assistant\n"
    completion = parts[1]

    # Принудительно меняем одинарные кавычки на двойные для валидности JSON
    completion = completion.replace("'", '"')

    return {
        "prompt": prompt,
        "completion": completion
    }

# Применяем ко всему датасету
train_dataset = train_dataset.map(prepare_prompt_completion)
val_dataset = val_dataset.map(prepare_prompt_completion)
#test_dataset = test_dataset.map(prepare_prompt_completion)

In [ ]:
### ДЕЛАЕМ ДАТАСЕТ ДЛЯ ОБУЧЕНИЯ
def prepare_train_dataset(df, tokenizer):
    """Функция для полной подготовки датасета к обучению"""

    def formatting_prompts_func(examples):
        instructions = []
        for text, target in zip(examples["text"], examples["json_target"]):

            user_prompt = (
                "Твоя задача — анализировать новости "
                "и определять динамику 18 показателей. Оценивай прямые упоминания и логические следствия. "
                "Допустимые значения изменений ТОЛЬКО ТАКИЕ: 'up', 'down', 'not stated'. "
                "Ответь строго в формате JSON без вступлений и пояснений: "
                "{"
                "\"процентная ставка\": \"тип изменения\","
                "\"ВВП\": \"тип изменения\","
                "\"инфляция\": \"тип изменения\","
                "\"безработица\": \"тип изменения\","
                "\"капитал\": \"тип изменения\","
                "\"инвестиции\": \"тип изменения\","
                "\"производство\": \"тип изменения\","
                "\"потребление\": \"тип изменения\","
                "\"численность рабочей силы\": \"тип изменения\","
                "\"сбережения\": \"тип изменения\","
                "\"заработные платы\": \"тип изменения\","
                "\"доходы населения\": \"тип изменения\","
                "\"валютный курс\": \"тип изменения\","
                "\"импорт\": \"тип изменения\","
                "\"экспорт\": \"тип изменения\","
                "\"государственные расходы\": \"тип изменения\","
                "\"государственный долг\": \"тип изменения\","
                "\"дефицит бюджета\": \"тип изменения\""
                "} "
                f"Текст новости: {text}"
            )

            messages = [
                {"role": "system", "content": "Ты — экспертный макроэкономический аналитик."},
                {"role": "user", "content": user_prompt}
            ]

            # Применяем CHAT TEMPLATE
            formatted_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            if isinstance(target, dict):
                target_str = json.dumps(target, ensure_ascii=False)
            else:
                target_str = str(target).strip()

            # СКЛЕИВАЕМ Промпт + Ответ + EOS токен
            full_text = f"{formatted_text}{target_str}{tokenizer.eos_token}"
            instructions.append(full_text)

        return {"text": instructions}

    # HuggingFace Dataset
    dataset = Dataset.from_pandas(df[['text', 'json_target']])

    dataset = dataset.map(formatting_prompts_func, batched=True)

    return dataset